In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

class TransformerBlock(layers.Layer):
    def __init__(self, dim, heads, ff_dim, **kwargs):
        super().__init__(**kwargs)   # VERY IMPORTANT
        self.dim = dim
        self.heads = heads
        self.ff_dim = ff_dim

        self.att = layers.MultiHeadAttention(
            num_heads=heads,
            key_dim=dim // heads
        )

        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(dim)
        ])

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x):
        attn = self.att(x, x)
        x = self.norm1(x + attn)
        ffn = self.ffn(x)
        return self.norm2(x + ffn)

    # REQUIRED for proper deserialization
    def get_config(self):
        config = super().get_config()
        config.update({
            "dim": self.dim,
            "heads": self.heads,
            "ff_dim": self.ff_dim,
        })
        return config

In [7]:
model = tf.keras.models.load_model(
    r"E:\project model ML\ISL.keras",
    custom_objects={"TransformerBlock": TransformerBlock},
    compile=False
)

print("Model Loaded Successfully")
model.summary()

E:\python3.10\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'transformer_block', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model Loaded Successfully


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 160, 160, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_160 (Functional)    │ (None, 5, 5, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape (Reshape)                    │ (None, 25, 1280)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block (TransformerBlock) │ (None, 25, 1280)            │       7,876,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 512)                 │         655,872 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 178)                 │          91,314 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,881,522 (41.51 MB)

 Trainable params: 8,623,538 (32.90 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


IMG_SIZE = model.input_shape[1]
print("Input size:", IMG_SIZE)

# ===============================
# 2️⃣ CLASS NAMES (178)
# ===============================

class_names = ['A LOT', 'ABUSE', 'ALL', 'ANGRY', 'ANY', 'ANYTHING', 'APPRECIATE',
'BEAUTIFUL', 'BED', 'BORED', 'BRING', 'CLASS', 'COLD', 'COLLEGE_SCHOOL', 'COMB',
'COME', 'CRYING', 'DARE', 'DIFFERENCE', 'DILEMMA', 'DISAPPOINTED', 'DO', "DON'T CARE",
'ENJOY', 'FAVOUR', 'FEVER', 'FINE', 'FOOD', 'FREE', 'FRIEND', 'GLASS', 'GO',
'GOOD', 'GOT', 'GRATEFUL', 'HAD', 'HAPPENED', 'HAPPY', 'HEAR', 'HEART',
'HELLO_HI', 'HELP', 'HIDING', 'HOW', 'HURT', 'I_ME_MINE_MY', 'KIND', 'KNOW',
'LEAVE', 'LIGHT', 'LIKE', 'LIKE_LOVE', 'MAKE', 'MEAN IT', 'MEDICINE', 'NAME',
'NEED', 'NEVER', 'NICE', 'NOT', 'NOW', 'NUMBER', 'OLD_AGE', 'ON THE WAY',
'ONWARDS', 'OUTSIDE', 'PHONE', 'PLACE', 'PLANNED', 'POUR', 'PREPARE', 'PROMISE',
'REALLY', 'REPEAT', 'ROOM', 'SERVE', 'SHIRT', 'SITTING', 'SLEEP', 'SLOWER',
'SO MUCH', 'SOFTLY', 'SOME HOW', 'SOME MORE', 'SOME ONE', 'SOMETHING', 'SORRY',
'SPEAK', 'STUBBORN', 'SURE', 'TAKE CARE', 'TAKE TIME', 'TALK', 'TELL', 'THANK',
'THAT', 'THERE', 'THINGS', 'THINK', 'THIS ONE', 'TIRED', 'TRAIN', 'TRUST',
'TRUTH', 'TURN ON', 'VERY', 'WANT', 'WATER', 'WEAR', 'WELCOME', 'WHAT', 'WHEN',
'WHO', 'WORRY', 'afraid', 'again', 'agree', 'answer', 'assistance', 'attendance',
'bad', 'become', 'book', 'break', 'careful', 'change', 'chat', 'college',
'congratulations', 'doctor', 'email', 'file', 'from', 'good morning',
'happy birthday', 'home', 'how are you', 'hungry', 'i need help', 'join',
'keepsmile', 'meet', 'mistake', 'open', 'opinion', 'pain', 'pass', 'please',
'practice', 'pray', 'pressure', 'problem', 'questions', 'remember', 'seat',
'secondary', 'shift', 'sick', 'skin', 'small', 'specific', 'stand', 'stop',
'sun', 'team', 'thirsty', 'this', 'today', 'together', 'understand', 'wait',
'warn', 'where', 'which', 'work', 'write', 'you']

# ===============================
# 3️⃣ START WEBCAM
# ===============================

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    # ROI box
    x1, y1 = 150, 100
    x2, y2 = 450, 400
    roi = frame[y1:y2, x1:x2]

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # ===============================
    # 4️⃣ PREPROCESS (MOBILE-NET STYLE)
    # ===============================

    img = cv2.resize(roi, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)   # VERY IMPORTANT

    # ===============================
    # 5️⃣ PREDICT
    # ===============================

    predictions = model.predict(img, verbose=0)

    class_index = np.argmax(predictions)
    confidence = np.max(predictions)

    predicted_label = class_names[class_index]

    # Confidence filter (important for 178 classes)
    if confidence < 0.65:
        predicted_label = "Detecting..."

    # ===============================
    # 6️⃣ DISPLAY
    # ===============================

    cv2.putText(frame,
                f"{predicted_label} ({confidence:.2f})",
                (150, 90),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2)

    cv2.imshow("Sign Language - MobileNet", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Input size: 160


IndexError: list index out of range

In [9]:
print("Model output shape:", model.output_shape)
print("Number of class names:", len(class_names))

Model output shape: (None, 178)
Number of class names: 177


In [11]:
import tensorflow as tf

IMG_SIZE = 160
BATCH_SIZE = 16

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "E:\project model ML\dataset_split/train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("Total classes:", len(class_names))

Found 14559 files belonging to 178 classes.
Total classes: 178


In [13]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "E:\project model ML\dataset_split/train",
    image_size=(160, 160),
    batch_size=16
)

class_names = train_ds.class_names

Found 14559 files belonging to 178 classes.


In [14]:
print("Model classes:", model.output_shape[1])
print("Dataset classes:", len(class_names))

Model classes: 178
Dataset classes: 178


In [16]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from collections import deque

IMG_SIZE = 160

# Load model (with custom TransformerBlock already defined)
model = tf.keras.models.load_model(
    r"E:\project model ML\ISL.keras",
    custom_objects={"TransformerBlock": TransformerBlock},
    compile=False
)

# Load class names properly
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "E:\project model ML\dataset_split/train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16
)
class_names = train_ds.class_names

# Smoothing buffer
prediction_buffer = deque(maxlen=8)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    x1, y1 = 150, 100
    x2, y2 = 450, 400
    roi = frame[y1:y2, x1:x2]
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

    img = cv2.resize(roi, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    predictions = model.predict(img, verbose=0)

    class_index = np.argmax(predictions)
    confidence = np.max(predictions)

    prediction_buffer.append(class_index)
    final_index = max(set(prediction_buffer), key=prediction_buffer.count)

    predicted_label = class_names[final_index]

    if confidence < 0.60:
        predicted_label = "Detecting..."

    cv2.putText(frame,
                f"{predicted_label} ({confidence:.2f})",
                (150, 90),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0,0,255),
                2)

    cv2.imshow("ISL Real-Time", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

E:\python3.10\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'transformer_block_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Found 14559 files belonging to 178 classes.
